# Analysing EEG Data with FieldTrip

**Author**: Judy D Zhu, Michèle Masson-Trottier

<div style="line-height: 2;">
<a href="https://github.com/JD-Zhu"><img src="https://img.shields.io/badge/-Judy_D_Zhu-181717?logo=github" alt="GitHub"></a><br>
<a href="https://github.com/micmas"><img src="https://img.shields.io/badge/-Michèle_Masson--Trottier-181717?logo=github" alt="GitHub"></a> <a href="https://orcid.org/0000-0002-0642-5662"><img src="https://img.shields.io/badge/ORCID-0000--0002--0642--5662-green?logo=orcid" alt="ORCID"></a>
</div>

**Date**: 30/03/2026

**License:**
<div style="margin-top: 10px;">
    <a href="https://creativecommons.org/licenses/by/4.0/" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> CC-BY-4.0 License
    </a>
</div>

## Purpose

This tutorial demonstrates how to use the compiled FieldTrip toolbox in Neurodesk to preprocess EEG data and compute event-related potentials (ERPs). A compiled version of FieldTrip is available in Neurodesk, allowing you to run FieldTrip analyses **without a MATLAB license**.

The dataset is a publicly available 64-channel EEG recording of a semantic categorisation task, distributed as part of the official [FieldTrip ERP tutorial](https://www.fieldtriptoolbox.org/tutorial/sensor/preprocessing_erp/). Participants judged nouns (e.g., *puppy*, *murderer*) as either affective or ontological, allowing comparison of ERP waveforms across task conditions.

:::{admonition} Learning Objectives
:class: tip
- Download an open EEG dataset and electrode layout from the FieldTrip server
- Open the FieldTrip container from the Neurodesk application menu
- Run a complete ERP analysis pipeline using the compiled FieldTrip runtime
- Understand how to inline trial function logic to work around the `addpath` limitation
- Inspect ERP plots and saved output figures
:::

## Citation and Resources

**FieldTrip:**
: Oostenveld, R., Fries, P., Maris, E., & Schoffelen, J.M. (2011). FieldTrip: Open Source Software for Advanced Analysis of MEG, EEG, and Invasive Electrophysiological Data. *Computational Intelligence and Neuroscience*, 2011, 156869. https://doi.org/10.1155/2011/156869

**Dataset:**
: FieldTrip ERP tutorial dataset (subject s04), distributed via https://download.fieldtriptoolbox.org/tutorial/preprocessing_erp/. Originally described in the FieldTrip tutorial on [Preprocessing of EEG data and computing ERPs](https://www.fieldtriptoolbox.org/tutorial/sensor/preprocessing_erp/).

**Educational Resources:**
: [FieldTrip website](https://www.fieldtriptoolbox.org/) | [FieldTrip ERP tutorial](https://www.fieldtriptoolbox.org/tutorial/sensor/preprocessing_erp/)

## Prerequisites

:::{warning}
Before starting this tutorial, ensure you have:
:::

- [ ] A running Neurodesk environment (Neurodesktop with graphical display)
- [ ] Basic familiarity with EEG concepts (epochs, ERPs, artefact rejection)
- [ ] No MATLAB license required - the compiled FieldTrip runtime is included in Neurodesk

:::{note}
**Compiled FieldTrip constraints:** The Neurodesk FieldTrip container uses a pre-compiled MATLAB runtime. Scripts can call any FieldTrip or SPM function, as well as MATLAB built-ins. However, **custom `.m` files cannot be added to the path** (`addpath` is not supported). This tutorial uses `ft_read_event` inline to replace the custom `trialfun_affcog` that the original FieldTrip tutorial relies on.
:::

## Step 1: Download the Dataset

Open a terminal in Neurodesktop and download the EEG data and electrode layout from the FieldTrip server:

```bash
mkdir -p ~/neurodesktop-storage/fieldtrip-erp/
cd ~/neurodesktop-storage/fieldtrip-erp/

wget https://download.fieldtriptoolbox.org/tutorial/preprocessing_erp/s04.eeg
wget https://download.fieldtriptoolbox.org/tutorial/preprocessing_erp/s04.vhdr
wget https://download.fieldtriptoolbox.org/tutorial/preprocessing_erp/s04.vmrk
wget https://download.fieldtriptoolbox.org/tutorial/preprocessing_erp/mpi_customized_acticap64.mat
```

The dataset is a 64-channel EEG recording in BrainVision format (~143 MB). The `.vhdr` file contains the channel and recording metadata, `.vmrk` the trigger markers, and `.eeg` the raw signal.

![Terminal showing wget download of the four FieldTrip ERP dataset files](/static/tutorials/electrophysiology/fieldtrip/download_terminal.png)
*Terminal output after downloading the four dataset files.*

## Step 2: Create the Analysis Script

Create a file called `fieldtrip_erp.m` in your working directory. Because the compiled FieldTrip container cannot load custom `.m` files via `addpath`, the trial-definition logic is written **inline** in the script using `ft_read_event` - a built-in FieldTrip function - rather than as a separate `trialfun` file.

```bash
cd ~/neurodesktop-storage/fieldtrip-erp/
nano fieldtrip_erp.m
```

Paste the following script:

```matlab
%% FieldTrip ERP tutorial - Neurodesk compiled version
%  Dataset: FieldTrip ERP tutorial data (s04)
%  https://www.fieldtriptoolbox.org/tutorial/sensor/preprocessing_erp/
%
%  Trigger codes:
%    S141 = word onset
%    S131 = affective judgment cue  (condition 1)
%    S132 = ontological judgment cue (condition 2)

%% --- 1. Build trial matrix (inline replacement for trialfun_affcog) ---

hdr   = ft_read_header('s04.vhdr');
event = ft_read_event('s04.vhdr');

% Extract sample indices and trigger values
EVsample = [event.sample]';
EVvalue  = {event.value}';

% Find all word-onset triggers (S141)
Word = find(strcmp('S141', EVvalue) == 1);

% Assign condition label from the following trigger
task = zeros(length(Word), 1);
for w = 1:length(Word)
    if Word(w)+1 <= length(EVvalue)
        if strcmp('S131', EVvalue{Word(w)+1})
            task(w) = 1;  % affective judgment
        elseif strcmp('S132', EVvalue{Word(w)+1})
            task(w) = 2;  % ontological judgment
        end
    end
end

% Remove trials with no condition label
valid    = task > 0;
Word     = Word(valid);
task     = task(valid);

% Define epoch window: 200 ms pre-stimulus, 1000 ms post-stimulus
PreTrig  = round(0.2 * hdr.Fs);
PostTrig = round(1.0 * hdr.Fs);

begsample = EVsample(Word) - PreTrig;
endsample = EVsample(Word) + PostTrig;
offset    = repmat(-PreTrig, length(Word), 1);

trl = [begsample endsample offset task];

%% --- 2. Preprocessing ---

cfg                 = [];
cfg.dataset         = 's04.eeg';
cfg.trl             = trl;
cfg.demean          = 'yes';
cfg.baselinewindow  = [-0.2 0];
cfg.lpfilter        = 'yes';
cfg.lpfreq          = 100;
cfg.implicitref     = 'LM';
cfg.reref           = 'yes';
cfg.refchannel      = {'LM' 'RM'};

data = ft_preprocessing(cfg);

%% --- 3. Compute bipolar EOG channels ---

% Vertical EOG
cfg         = [];
cfg.channel = {'53' 'LEOG'};
cfg.reref   = 'yes';
cfg.refchannel = {'53'};
eogv        = ft_preprocessing(cfg, data);
cfg         = []; cfg.channel = 'LEOG';
eogv        = ft_selectdata(cfg, eogv);
eogv.label  = {'eogv'};

% Horizontal EOG
cfg         = [];
cfg.channel = {'57' '25'};
cfg.reref   = 'yes';
cfg.refchannel = {'57'};
eogh        = ft_preprocessing(cfg, data);
cfg         = []; cfg.channel = '25';
eogh        = ft_selectdata(cfg, eogh);
eogh.label  = {'eogh'};

% Remove original EOG channels, append bipolar versions
cfg         = []; cfg.channel = setdiff(1:60, [25 53 57]);
data        = ft_selectdata(cfg, data);
cfg         = [];
data        = ft_appenddata(cfg, data, eogv, eogh);

%% --- 4. Artefact rejection (visual summary mode) ---

cfg             = [];
cfg.method      = 'summary';
cfg.layout      = 'mpi_customized_acticap64.mat';
cfg.channel     = 1:60;
data_clean      = ft_rejectvisual(cfg, data);

%% --- 5. ERP computation ---

cfg        = []; cfg.trials = find(data_clean.trialinfo == 1);
erp_task1  = ft_timelockanalysis(cfg, data_clean);   % affective

cfg        = []; cfg.trials = find(data_clean.trialinfo == 2);
erp_task2  = ft_timelockanalysis(cfg, data_clean);   % ontological

%% --- 6. Difference wave ---

cfg            = [];
cfg.operation  = 'subtract';
cfg.parameter  = 'avg';
difference     = ft_math(cfg, erp_task1, erp_task2);

%% --- 7. Plot and save results ---

% Multi-channel ERP plot
% Note: cfg.interactive = 'yes' opens a clickable GUI.
% The print() command below will only execute after you CLOSE the figure window.
cfg              = [];
cfg.layout       = 'mpi_customized_acticap64.mat';
cfg.interactive  = 'yes';
cfg.showoutline  = 'yes';
ft_multiplotER(cfg, erp_task1, erp_task2);
print(gcf, '-dpng', 'erp_multiplot.png');

% Topoplot of difference wave at 300-500 ms
cfg              = [];
cfg.layout       = 'mpi_customized_acticap64.mat';
cfg.xlim         = [0.3 0.5];
ft_topoplotER(cfg, difference);
print(gcf, '-dpng', 'erp_topo_diff_300_500ms.png');

disp('Done! Figures saved to erp_multiplot.png and erp_topo_diff_300_500ms.png');
```

![Script open in nano text editor in the Neurodesktop terminal](/static/tutorials/electrophysiology/fieldtrip/script_nano.png)
*The `fieldtrip_erp.m` script open in the nano editor.*

## Step 3: Open the FieldTrip Container

Navigate to the Neurodesk application menu:

**Applications → Neurodesk → Electrophysiology → fieldtrip → fieldtrip20220617**

![Opening FieldTrip from the Neurodesk application menu](/static/tutorials/electrophysiology/fieldtrip/1_menu.png)
*Opening FieldTrip from the Neurodesk application menu.*

This launches a terminal window running inside the FieldTrip Singularity container:

![The FieldTrip container terminal window ready to use](/static/tutorials/electrophysiology/fieldtrip/2_container.png)
*The FieldTrip container terminal, ready to run scripts.*

## Step 4: Run the Analysis

In the FieldTrip container terminal, navigate to your working directory and run the script:

```bash
cd ~/neurodesktop-storage/fieldtrip-erp/
run_fieldtrip.sh /opt/MCR/v99 ./fieldtrip_erp.m
```

The script will:
1. Read the EEG header and event markers to build the trial matrix
2. Preprocess data (baseline correction, 100 Hz low-pass filter, re-reference to linked mastoids)
3. Compute bipolar EOG channels
4. Open the interactive **summary artefact rejection** window - drag to select outlier trials, then close the window to continue
5. Compute ERPs for each condition and a difference wave
6. Save two figures: `erp_multiplot.png` and `erp_topo_diff_300_500ms.png`

![Terminal output while running the FieldTrip ERP script](/static/tutorials/electrophysiology/fieldtrip/3_running.png)
*Terminal output while running the `fieldtrip_erp.m` script with the compiled runtime.*

:::{note}
The compiled MATLAB runtime takes a minute or two to initialise on first run. Subsequent runs are faster.
:::

## Step 5: Artefact Rejection

When `ft_rejectvisual` opens, you will see a summary window showing all channels and trials. Trials or channels with unusually high variance are highlighted.

![ft_rejectvisual summary mode showing trials and channels](/static/tutorials/electrophysiology/fieldtrip/4_rejectvisual.png)
*The `ft_rejectvisual` summary window. Each square represents one trial; colour indicates signal variance.*

- In the **bottom-left scatter plot**, drag a selection box around the outlier dots (those well above the main cluster) to mark trials for removal
- Suggested trials to remove: **22, 42, 89, 90, 92, 126, 136, 150** (from the official FieldTrip tutorial)
- Close the window when done - the script continues automatically with the clean data

## Step 6: Inspect the Results

After the script finishes, two PNG files will be saved in `~/neurodesktop-storage/fieldtrip-erp/`.

### Multi-channel ERP plot

```bash
ls -lh ~/neurodesktop-storage/fieldtrip-erp/*.png
```

:::{note}
Run this in a regular Neurodesktop terminal (not the FieldTrip container terminal) - or simply open the `fieldtrip-erp` folder in the file manager.
:::

![fieldtrip-erp directory content](/static/tutorials/electrophysiology/fieldtrip/fieldtrip-erp-folder.png)
*The `fieldtrip-erp` folder content showing the 2 produced PNGs.*

Open `erp_multiplot.png` to see all 60 EEG channels. You may need to tell Neurodesktop to use the `Image Viewer` to open the file. Each panel shows the ERP for the affective (blue) and ontological (orange) conditions:

![Multi-channel ERP plot showing task1 vs task2 across all 64 electrodes](/static/tutorials/electrophysiology/fieldtrip/5_erp_multiplot.png)
*Multi-channel ERP plot comparing the affective and ontological judgment conditions.*

### Difference wave topography

Open `erp_topo_diff_300_500ms.png` to see the scalp topography of the difference wave (affective minus ontological) averaged over the 300–500 ms window:

![Topoplot of the ERP difference wave between 300 and 500 ms](/static/tutorials/electrophysiology/fieldtrip/6_erp_topo.png)
*Scalp topography of the difference wave at 300–500 ms post-stimulus.*

## Understanding the Trial Function Workaround

The official FieldTrip tutorial uses a custom file `trialfun_affcog.m` passed to `cfg.trialfun`. This **cannot** be used with compiled FieldTrip in Neurodesk because custom `.m` files cannot be added to the search path.

The workaround used here replaces that file by calling `ft_read_event` directly inside the script and constructing the `cfg.trl` matrix manually before calling `ft_preprocessing`. This achieves the same result:

| Approach | Official tutorial | This tutorial |
|---|---|---|
| Trial definition | `cfg.trialfun = 'trialfun_affcog'` | `ft_read_event` + inline loop |
| Requires custom `.m` | Yes | No |
| Works with compiled FieldTrip | No | **Yes** |
| Analysis result | Identical | Identical |

This same pattern - reading events inline and building `cfg.trl` manually - can be applied to any dataset when adapting existing FieldTrip scripts for the compiled Neurodesk container.

## Summary

1. Downloaded the FieldTrip ERP tutorial dataset (BrainVision format, ~143 MB) from the FieldTrip server
2. Wrote a standalone `fieldtrip_erp.m` script that inlines trial-definition logic using `ft_read_event` to avoid the `addpath` limitation of compiled FieldTrip
3. Opened the FieldTrip container from the Neurodesk application menu
4. Ran the complete ERP pipeline with `run_fieldtrip.sh`: preprocessing → EOG computation → artefact rejection → ERP averaging → difference wave → figure export
5. Inspected multi-channel ERP plots and a scalp topography of the 300–500 ms difference wave

:::{seealso}
- [Analysing EEG Data with MNE-Python](../../examples/electrophysiology/eeg_with_mne.ipynb) - Python-based alternative using the MNE toolbox
- [FieldTrip ERP tutorial](https://www.fieldtriptoolbox.org/tutorial/sensor/preprocessing_erp/) - Full official tutorial with additional steps (time-frequency, source analysis)
:::